# Import tools
Simulation, Data Manipulation and Math Operations

In [1]:
# Importing libraries & frameworks 
import random 
import genesis as gs
import numpy as np
import torch.nn.functional as F
import cv2
import json
from pathlib import Path

[I 05/07/25 12:50:23.771 1250768] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


# Initializing Genesis

Below is defined some setup the scene and camera as well as the definition of the Rigid Body Entities used in the Grasp (Bottle and the Spot Gripper). 


In [2]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    precision='32',
    debug=False,
    eps=1e-12,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    vis_options=gs.options.VisOptions(show_world_frame=False),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=False, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())

# bottle_radius = 0.025
bottle_pos = [0, 0.0, 0.1] 

plastic_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="../bottles/plastic_bottle/bottle.stl", 
        pos=bottle_pos,            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.0002                      
    )
)

# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = (0.0, 0.0, 0.09),
    res    = (640, 480),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)
# Building the Scene
scene.build()

[Genesis] [12:50:26] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [12:50:26] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [12:50:26] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [12:50:26] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [12:50:26] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [12:50:27] [INFO] Scene <cf10cf6> created.
[Genesis] [12:50:27] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <a0da2f3>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [12:50:27] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <526063d>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/spot_gen/Exp_RigidEntity/bottles/plastic_bottle/bottle.stl')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [12:50:28] [INFO] Building scene <cf10cf6>...
[Genesis] [12:50:31] [INFO] Compiling simulation kernels...
[Genesis] [12:50:36] [INFO] Building visualizer...


## Pre-dataset Parameters

Below some parameters for the dataset creation are defined.

In [5]:
# Output directory
dataset_dir = Path("../water_bottle_dataset3")
dataset_dir.mkdir(parents=True, exist_ok=True)

# Verification directory
image_dir = Path("../img_verification_dataset3")
image_dir.mkdir(parents=True, exist_ok=True)

up = [0, 0, 1]
# Number of points in the grid
nx_points, nz_points = 100, 10

# Create a 2D grid with the given specifications 
x = np.linspace(start=-0.5, stop=0.5, num=nx_points)
z = np.linspace(start=0.16, stop=0.6, num=nz_points)
X, Z = np.meshgrid(x, z)
Y = np.full_like(X, 0.5)  # y fixed at 0.5

# Stack into (x, y, z) points
grid_points = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

print(f"Grid points: {grid_points.shape}")  # Outputs (1_000, 3)


Grid points: (1000, 3)


## Image extraction Loop

In this loop all the randomization, robot action, data save happens.

In [6]:
for i, (x, y, z) in enumerate(grid_points):
    # Randomize camera lookat
    rand_lookat_x = random.uniform(-0.03, 0.03)
    rand_lookat_y = random.uniform(-0.03, 0.03)
    rand_lookat_z = random.uniform(0, 0.09)   

    cam_lookat = [rand_lookat_x, rand_lookat_y, rand_lookat_z]
    cam_pos = [x, y, z]

    # Change camera position
    cam.set_pose(
        pos= cam_pos,
        lookat= cam_lookat
    )
    
    #  View setup 
    view_id = f"view_{i}"  # Ensure consistent naming 
    view_dir = dataset_dir / view_id
    img_view_dir = image_dir / view_id
    view_dir.mkdir(parents=True, exist_ok=True)
    img_view_dir.mkdir(parents=True, exist_ok=True)

    # Get RGB-D data
    for x in range(5):
        scene.step()
        render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False)

    rgb, depth, seg, normal_map = render_output
    bottle_pixels = np.where(np.array(seg) == 1)
    bottle_pixels = np.transpose(bottle_pixels)  # Shape: (num_pixels, 2)

    # Save data
    np.save(view_dir / "rgb.npy", rgb)
    np.save(view_dir / "depth.npy", depth)
    np.save(view_dir / "segmentation.npy", seg)
    np.save(view_dir / "normal.npy", normal_map)

    # Save PNGs for visual verification
    cv2.imwrite(str(img_view_dir / "rgb.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

    # Store metadata
    data_metadata = {
        "view_id": view_id,
        "camera_pos": cam_pos,
        "camera_lookat": cam_lookat,
        "up": up
    }

    # Save metadata
    with open(view_dir / "metadata.json", "w") as f:
        json.dump(data_metadata, f, indent=4)

    for _ in range(200):
        scene.step()

Bottler pixels quantity: 12365
Bottler pixels quantity: 12935
Bottler pixels quantity: 13218
Bottler pixels quantity: 13080
Bottler pixels quantity: 13716
Bottler pixels quantity: 14095
Bottler pixels quantity: 14652
Bottler pixels quantity: 14560
Bottler pixels quantity: 14317
Bottler pixels quantity: 15448
Bottler pixels quantity: 15310
Bottler pixels quantity: 16596
Bottler pixels quantity: 15404
Bottler pixels quantity: 16247
Bottler pixels quantity: 15711
Bottler pixels quantity: 17212
Bottler pixels quantity: 18336
Bottler pixels quantity: 17761
Bottler pixels quantity: 16877
Bottler pixels quantity: 16496
Bottler pixels quantity: 17080
Bottler pixels quantity: 14109
Bottler pixels quantity: 18902
Bottler pixels quantity: 20527
Bottler pixels quantity: 20212
Bottler pixels quantity: 19858
Bottler pixels quantity: 20997
Bottler pixels quantity: 20982
Bottler pixels quantity: 21551
Bottler pixels quantity: 15867
Bottler pixels quantity: 16784
Bottler pixels quantity: 21106
Bottler 